#Bradly-Terry

###imports

In [3]:
!pip install pymc arviz
import pandas as pd
import glob
import os
import re

###preprocess data

In [2]:
def extract_dataset(filename):
    """Извлекает название датасета из имени файла (оставляет только ML-1M, Amazon Baby, Amazon Toys)."""
    base = os.path.splitext(filename)[0]
    # Удаляем префикс до последнего тире с пробелом и суффикс _kNN
    # Пример: "experimental_results.xlsx - ML-1M_k10" -> "ML-1M"
    match = re.search(r'[-–]\s*([A-Za-z0-9\s\-]+)_k\d+$', base)
    if match:
        return match.group(1).strip()
    # fallback
    base_clean = re.sub(r'_k\d+$', '', base)
    if ' - ' in base_clean:
        return base_clean.split(' - ')[-1].strip()
    return base_clean

def find_column(df, possible_names):
    """Ищет столбец, имя которого содержит одну из подстрок (регистронезависимо)."""
    for col in df.columns:
        col_lower = col.lower().strip()
        for name in possible_names:
            if name.lower() in col_lower:
                return col
    return None

def process_file(filepath):
    """Загружает один файл и преобразует его в формат (dataset, model, score)."""
    ext = os.path.splitext(filepath)[1].lower()

    # Для CSV пробуем разные разделители, пропускаем 1 строку, заголовок со 2-й строки
    if ext == '.csv':
        for sep in [',', '\t', ';']:
            try:
                df = pd.read_csv(filepath, sep=sep, skiprows=1, encoding='utf-8')
                if len(df.columns) > 1:
                    break
            except:
                continue
        else:
            df = pd.read_csv(filepath, skiprows=1, encoding='utf-8')
    elif ext == '.xlsx':
        df = pd.read_excel(filepath, skiprows=1)
    else:
        raise ValueError(f"Неподдерживаемый формат: {ext}")

    # Поиск нужных столбцов
    model_col = find_column(df, ['model'])
    maxlen_col = find_column(df, ['maxlen', 'max len'])
    recall_col = find_column(df, ['recall'])

    if model_col is None or maxlen_col is None or recall_col is None:
        available = list(df.columns)
        raise ValueError(f"Не найдены столбцы. Доступные: {available}. "
                         f"Model: {model_col}, Maxlen: {maxlen_col}, Recall: {recall_col}")

    # Переименовываем для единообразия
    df = df.rename(columns={model_col: 'Model', maxlen_col: 'Maxlen', recall_col: 'Recall'})

    # Извлекаем название датасета из имени файла
    dataset = extract_dataset(os.path.basename(filepath))

    # Создаём новое имя модели (склеиваем Model и Maxlen, если Maxlen != '-')
    def make_model_name(row):
        if row['Maxlen'] != '-' and pd.notna(row['Maxlen']):
            return f"{row['Model']} {row['Maxlen']}"
        return row['Model']

    df['model'] = df.apply(make_model_name, axis=1)

    # Формируем итоговый датафрейм
    result = df[['model', 'Recall']].copy()
    result['dataset'] = dataset
    result = result.rename(columns={'Recall': 'score'})
    result = result[['dataset', 'model', 'score']]
    return result

def main():
    k_values = [10, 20, 100]
    for k in k_values:
        pattern = f"*_k{k}.*"
        files = glob.glob(pattern)
        if not files:
            print(f"Файлы для k={k} не найдены")
            continue

        all_dfs = []
        for f in files:
            try:
                all_dfs.append(process_file(f))
            except Exception as e:
                print(f"Ошибка при обработке {f}: {e}")

        if all_dfs:
            combined = pd.concat(all_dfs, ignore_index=True)
            out_file = f"combined_k{k}.csv"
            combined.to_csv(out_file, index=False)
            print(f"Сохранён {out_file} ({len(combined)} записей)")
        else:
            print(f"Для k={k} нет корректных данных")

if __name__ == "__main__":
    main()

Сохранён combined_k10.csv (33 записей)
Сохранён combined_k20.csv (33 записей)
Сохранён combined_k100.csv (33 записей)


###Запуск ранжирования

In [5]:
!python paper_bt.py

Multiprocess sampling (4 chains in 4 jobs)
CompoundStep
>Metropolis: [sigma_bar]
>Metropolis: [beta]
Sampling 4 chains for 5_000 tune and 20_000 draw iterations (20_000 + 80_000 draws total) took 97 seconds.

Файл: combined_k10.csv
Финальный рейтинг по posterior mean beta:
 rank        model  beta_mean  beta_2.5%  beta_97.5%  weight_mean  weight_2.5%  weight_97.5%  total_wins
    1 DiffuRec 100   4.019991   1.873446    6.890704     0.530505     0.174577      0.897301   29.000000
    2  DiffuRec 50   3.549206   1.524392    6.156915     0.357190     0.068523      0.744534   28.000000
    3    ADRec 100   1.403648  -0.253195    3.225660     0.052245     0.003381      0.164032   22.000000
    4     ADRec 50   0.075231  -1.594735    1.738299     0.014881     0.000756      0.051618   16.000000
    5   SASRec 100  -0.117454  -1.796893    1.500546     0.012367     0.000608      0.043200   15.000000
    6    SASRec 50  -0.303322  -1.983190    1.309857     0.010376     0.000485      0.036853   1